# 02 — Preprocessing

Nettoyage, fusion des tables, encodage et feature engineering.
Resultat attendu : un DataFrame propre pret pour l entrainement.

## Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

# nrows=10000 pendant le developpement — retirer pour l entrainement final
NROWS = 10000

## 1. Chargement des tables

In [2]:
app_train = pd.read_csv("../data/application_train.csv", nrows=NROWS)
app_test  = pd.read_csv("../data/application_test.csv",  nrows=NROWS)
print("app_train :", app_train.shape)
print("app_test  :", app_test.shape)

app_train : (10000, 122)
app_test  : (10000, 121)


In [3]:
bureau       = pd.read_csv("../data/bureau.csv",               nrows=NROWS)
prev_app     = pd.read_csv("../data/previous_application.csv", nrows=NROWS)
installments = pd.read_csv("../data/installments_payments.csv",nrows=NROWS)
print("Tables secondaires chargees")

Tables secondaires chargees


## 2. Nettoyage de application_train / test

### 2.1 Anomalie DAYS_EMPLOYED

Valeur 365243 = valeur codee signifiant "inconnu" (detectee en EDA). On la remplace par NaN et on cree un flag.

In [4]:
for df in [app_train, app_test]:
    df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
    df["DAYS_EMPLOYED"].replace({365243: np.nan}, inplace=True)

print("Anomalies train :", app_train["DAYS_EMPLOYED_ANOM"].sum())
print("Anomalies test  :", app_test["DAYS_EMPLOYED_ANOM"].sum())

Anomalies train : 1774
Anomalies test  : 1883


### 2.2 Suppression colonnes avec >60% de valeurs manquantes

Justification : trop peu d informations exploitables pour imputer de maniere fiable.

In [5]:
threshold = 0.6
missing_rate = app_train.isnull().mean()
cols_to_drop = missing_rate[missing_rate > threshold].index.tolist()
print(f"{len(cols_to_drop)} colonnes supprimees (>{int(threshold*100)}% manquants) :")
print(cols_to_drop)

app_train.drop(columns=cols_to_drop, inplace=True)
app_test.drop(columns=[c for c in cols_to_drop if c in app_test.columns], inplace=True)

17 colonnes supprimees (>60% manquants) :
['OWN_CAR_AGE', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'FLOORSMIN_AVG', 'LIVINGAPARTMENTS_AVG', 'NONLIVINGAPARTMENTS_AVG', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'FLOORSMIN_MODE', 'LIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_MODE', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'FLOORSMIN_MEDI', 'LIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'FONDKAPREMONT_MODE']


### 2.3 Imputation des valeurs manquantes restantes

Numeriques -> mediane (robuste aux outliers). Categorielles -> mode.

In [6]:
num_cols = app_train.select_dtypes(include=["number"]).columns
for col in num_cols:
    median = app_train[col].median()
    app_train[col].fillna(median, inplace=True)
    if col in app_test.columns:
        app_test[col].fillna(median, inplace=True)

cat_cols = app_train.select_dtypes(include=["object"]).columns
for col in cat_cols:
    mode = app_train[col].mode()[0]
    app_train[col].fillna(mode, inplace=True)
    if col in app_test.columns:
        app_test[col].fillna(mode, inplace=True)

print("Manquants restants train :", app_train.isnull().sum().sum())
print("Manquants restants test  :", app_test.isnull().sum().sum())

Manquants restants train : 0
Manquants restants test  : 0


## 3. Feature engineering — tables secondaires

On agreuge chaque table en une ligne par client (SK_ID_CURR) puis on la joint a app_train/test.

### 3.1 bureau.csv

In [7]:
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_COUNT           = ("SK_ID_BUREAU", "count"),
    BUREAU_CREDIT_SUM_MEAN = ("AMT_CREDIT_SUM", "mean"),
    BUREAU_CREDIT_SUM_MAX  = ("AMT_CREDIT_SUM", "max"),
    BUREAU_DAYS_CREDIT_MEAN= ("DAYS_CREDIT", "mean"),
).reset_index()
print("bureau_agg shape :", bureau_agg.shape)
bureau_agg.head()

bureau_agg shape : (2011, 5)


,SK_ID_CURR,BUREAU_COUNT,BUREAU_CREDIT_SUM_MEAN,BUREAU_CREDIT_SUM_MAX,BUREAU_DAYS_CREDIT_MEAN
0,100053,7,92655.642857,225000.0,-2287.714286
1,100568,7,119269.877143,333000.0,-493.857143
2,100653,7,124783.200000,315000.0,-1237.571429
3,100802,1,157500.000000,157500.0,-525.000000
4,100819,18,232115.200000,720000.0,-917.555556


### 3.2 previous_application.csv

In [8]:
prev_agg = prev_app.groupby("SK_ID_CURR").agg(
    PREV_COUNT           = ("SK_ID_PREV", "count"),
    PREV_AMT_CREDIT_MEAN = ("AMT_CREDIT", "mean"),
    PREV_AMT_ANNUITY_MEAN= ("AMT_ANNUITY", "mean"),
).reset_index()
print("prev_agg shape :", prev_agg.shape)
prev_agg.head()

prev_agg shape : (9734, 4)


,SK_ID_CURR,PREV_COUNT,PREV_AMT_CREDIT_MEAN,PREV_AMT_ANNUITY_MEAN
0,100009,1,98239.5,8996.760
1,100035,1,0.0,NaN
2,100043,1,33340.5,5287.815
3,100067,1,450000.0,16164.000
4,100077,1,0.0,NaN


### 3.3 installments_payments.csv — features de retard de paiement

In [9]:
# Retard = difference entre montant du au montant paye
installments["PAYMENT_DIFF"] = installments["AMT_INSTALMENT"] - installments["AMT_PAYMENT"]
installments["DAYS_LATE"]    = installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]

install_agg = installments.groupby("SK_ID_CURR").agg(
    INSTALL_COUNT             = ("SK_ID_PREV", "count"),
    INSTALL_PAYMENT_DIFF_MEAN = ("PAYMENT_DIFF", "mean"),
    INSTALL_DAYS_LATE_MEAN    = ("DAYS_LATE", "mean"),
    INSTALL_DAYS_LATE_MAX     = ("DAYS_LATE", "max"),
).reset_index()
print("install_agg shape :", install_agg.shape)
install_agg.head()

install_agg shape : (8893, 5)


,SK_ID_CURR,INSTALL_COUNT,INSTALL_PAYMENT_DIFF_MEAN,INSTALL_DAYS_LATE_MEAN,INSTALL_DAYS_LATE_MAX
0,100009,1,0.000,-13.0,-13.0
1,100011,1,14138.865,958.0,958.0
2,100012,1,0.000,-41.0,-41.0
3,100035,1,0.000,-3.0,-3.0
4,100042,1,157.500,-13.0,-13.0


## 4. Fusion de toutes les tables

In [10]:
def merge_all(df):
    df = df.merge(bureau_agg,  on="SK_ID_CURR", how="left")
    df = df.merge(prev_agg,    on="SK_ID_CURR", how="left")
    df = df.merge(install_agg, on="SK_ID_CURR", how="left")
    return df

app_train = merge_all(app_train)
app_test  = merge_all(app_test)

print("app_train apres fusion :", app_train.shape)
print("app_test  apres fusion :", app_test.shape)

app_train apres fusion : (10000, 117)
app_test  apres fusion : (10000, 116)


In [11]:
# Les clients sans historique dans les tables secondaires ont des NaN apres le left join
# On les impute a 0 (pas de credit = 0)
new_cols = (
    bureau_agg.columns.tolist()[1:] +
    prev_agg.columns.tolist()[1:] +
    install_agg.columns.tolist()[1:]
)
for col in new_cols:
    if col in app_train.columns:
        app_train[col].fillna(0, inplace=True)
    if col in app_test.columns:
        app_test[col].fillna(0, inplace=True)

print("Manquants apres fusion :", app_train.isnull().sum().sum())

Manquants apres fusion : 0


## 5. Encodage des variables categorielles

- **LabelEncoding** pour les variables binaires (<=2 categories)
- **OneHotEncoding** pour les variables nominales (>2 categories)

In [12]:
le = LabelEncoder()
le_count = 0

for col in app_train.select_dtypes("object").columns:
    if app_train[col].nunique() <= 2:
        le.fit(app_train[col])
        app_train[col] = le.transform(app_train[col])
        app_test[col]  = le.transform(app_test[col])
        le_count += 1

print(f"{le_count} colonnes label-encodees")

5 colonnes label-encodees


In [13]:
app_train = pd.get_dummies(app_train)
app_test  = pd.get_dummies(app_test)

# Alignement : garder uniquement les colonnes presentes dans les deux DataFrames
train_labels = app_train["TARGET"]
app_train, app_test = app_train.align(app_test, join="inner", axis=1)
app_train["TARGET"] = train_labels

print("app_train apres encodage :", app_train.shape)
print("app_test  apres encodage :", app_test.shape)

app_train apres encodage : (10000, 227)
app_test  apres encodage : (10000, 226)


## 6. Sauvegarde du dataset final

In [14]:
app_train.to_csv("../data/train_preprocessed.csv", index=False)
app_test.to_csv("../data/test_preprocessed.csv",   index=False)
print("Sauvegardes :")
print("  train_preprocessed.csv :", app_train.shape)
print("  test_preprocessed.csv  :", app_test.shape)

Sauvegardes :
  train_preprocessed.csv : (10000, 227)
  test_preprocessed.csv  : (10000, 226)


## Conclusion

- Tables fusionnees : application_train + bureau + previous_application + installments_payments
- Anomalie DAYS_EMPLOYED traitée (flag + remplacement NaN)
- Colonnes >60% manquants supprimées
- Imputation : mediane (numerique), mode (catégoriel)
- Encodage : LabelEncoding (binaire) + OneHotEncoding (nominal) + alignement train/test
- Dataset pret pour 03_modeling.ipynb